In [4]:
import pandas as pd

matches = pd.read_csv('../data/processed/matches_clean.csv')
history = pd.read_csv('../data/processed/elo_history.csv')

In [5]:
merged_df = pd.merge(matches, history, on=['home_team','away_team','date'], how='inner')
merged_df.shape

(49217, 20)

In [6]:
merged_df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral_x', 'result_x',
       'match_importance_x', 'winner', 'home_elo_pre', 'away_elo_pre',
       'home_elo_post', 'away_elo_post', 'result_y', 'neutral_y',
       'match_importance_y', 'elo_diff'],
      dtype='str')

In [7]:
merged_df = merged_df.rename(columns={"neutral_x": "neutral", "match_importance_x": "match_importance"})
clean_df = merged_df[['home_score', 'away_score', 'elo_diff', 'neutral', 'match_importance', 'date']]

In [8]:
clean_df.head()

,home_score,away_score,elo_diff,neutral,match_importance,date
0,0.0,0.0,0.000000,False,FRIENDLY,1872-11-30
1,4.0,2.0,0.000000,False,FRIENDLY,1873-03-08
2,2.0,1.0,-9.490296,False,FRIENDLY,1874-03-07
3,2.0,2.0,3.291652,False,FRIENDLY,1875-03-06
4,3.0,0.0,-3.291652,False,FRIENDLY,1876-03-04


In [9]:
import sys
sys.path.append('../src')
from config import K_FACTORS

clean_df['match_importance'] = clean_df['match_importance'].map(K_FACTORS)

In [10]:
from sklearn.linear_model import PoissonRegressor

clean_train = clean_df[clean_df['date'] < '2020-01-01']
clean_test = clean_df[clean_df['date'] >= '2020-01-01']

In [11]:
X_train = clean_train[['elo_diff', 'neutral', 'match_importance']]
X_test = clean_test[['elo_diff', 'neutral', 'match_importance']]

y_train_home = clean_train['home_score']
y_train_away = clean_train['away_score']

y_test_home = clean_test['home_score']
y_test_away = clean_test['away_score']

In [12]:
model_home = PoissonRegressor()
model_home.fit(X_train, y_train_home)

model_away = PoissonRegressor()
model_away.fit(X_train, y_train_away)

,"alpha alpha: float, default=1Constant that multiplies the L2 penalty term and determines theregularization strength. ``alpha = 0`` is equivalent to unpenalizedGLMs. In this case, the design matrix `X` must have full column rank(no collinearities).Values of `alpha` must be in the range `[0.0, inf)`.",1.0
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the linear predictor (`X @ coef + intercept`).",True
,"solver solver: {'lbfgs', 'newton-cholesky'}, default='lbfgs'Algorithm to use in the optimization problem:'lbfgs' Calls scipy's L-BFGS-B optimizer.'newton-cholesky' Uses Newton-Raphson steps (in arbitrary precision arithmetic equivalent to iterated reweighted least squares) with an inner Cholesky based solver. This solver is a good choice for `n_samples` >> `n_features`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features` because it explicitly computes the Hessian matrix. .. versionadded:: 1.2",'lbfgs'
,"max_iter max_iter: int, default=100The maximal number of iterations for the solver.Values must be in the range `[1, inf)`.",100
,"tol tol: float, default=1e-4Stopping criterion. For the lbfgs solver,the iteration will stop when ``max{|g_j|, j = 1, ..., d} <= tol``where ``g_j`` is the j-th component of the gradient (derivative) ofthe objective function.Values must be in the range `(0.0, inf)`.",0.0001
,"warm_start warm_start: bool, default=FalseIf set to ``True``, reuse the solution of the previous call to ``fit``as initialization for ``coef_`` and ``intercept_`` .",False
,"verbose verbose: int, default=0For the lbfgs solver set verbose to any positive number for verbosity.Values must be in the range `[0, inf)`.",0
Name,Type,Value
"coef_ coef_: array of shape (n_features,)Estimated coefficients for the linear predictor (`X @ coef_ +intercept_`) in the GLM.","ndarray[float64](3,)","[0.,0.,0.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](3,)","['elo_diff','neutral','match_importance']"
intercept_ intercept_: floatIntercept (a.k.a. bias) added to linear predictor.,float64,0.1756


In [13]:
from sklearn.metrics import mean_absolute_error

pred_home = model_home.predict(X_test)
pred_away = model_away.predict(X_test)

print(f"MAE home: {mean_absolute_error(y_test_home, pred_home):.4f}")
print(f"MAE away: {mean_absolute_error(y_test_away, pred_away):.4f}")

MAE home: 1.2425
MAE away: 0.9674


In [15]:
import joblib

joblib.dump(model_home, '../models/model_home.pkl')
joblib.dump(model_away, '../models/model_away.pkl')

['../models/model_away.pkl']

In [16]:
import pandas as pd

fixtures = pd.read_csv('../data/raw/wc2026_fixtures.csv')
elo = pd.read_csv('../data/processed/elo_current.csv')

fixture_teams = set(fixtures['Home Team'].tolist() + fixtures['Away Team'].tolist())
elo_teams = set(elo['team'].tolist())

no_match = fixture_teams - elo_teams
print(no_match)

{'3CEFHI', '1H', '3ABCDF', '2B', '2J', '2E', '1D', 'To be announced', 'Korea Republic', 'Türkiye', '3EFGIJ', '1E', '1K', '2H', '3CDFGH', '1B', '3DEIJL', '2G', '1I', '2C', 'Czechia', '1F', '3EHIJK', '1J', 'Cabo Verde', '2F', '1A', 'IR Iran', '2A', '1C', 'Congo DR', '3BEFIJ', "Côte d'Ivoire", '1L', '2D', '2I', '3AEHIJ', '2L', '2K', 'USA', '1G'}


In [17]:
real_missing = {t for t in no_match if not any(c.isdigit() for c in t) and t != 'To be announced'}
print(real_missing)

{'Congo DR', 'Czechia', "Côte d'Ivoire", 'Cabo Verde', 'IR Iran', 'USA', 'Korea Republic', 'Türkiye'}


In [18]:
for team in real_missing:
    print(team, "→", elo[elo['team'].str.contains(team.split()[0], case=False)]['team'].tolist())

Congo DR → ['DR Congo', 'Congo']
Czechia → []
Côte d'Ivoire → []
Cabo Verde → []
IR Iran → ['Northern Ireland', 'Republic of Ireland', 'Iran', 'Iraq', 'United Arab Emirates', 'Kiribati', 'British Virgin Islands', 'United States Virgin Islands', 'Iraqi Kurdistan', 'Bonaire', 'Yorkshire']
USA → []
Korea Republic → ['South Korea', 'North Korea', 'United Koreans in Japan']
Türkiye → []


In [19]:
check = ['Czech Republic', 'Turkey', 'United States', 'Ivory Coast', 'Cape Verde']
for t in check:
    print(t, t in elo_teams)

Czech Republic True
Turkey True
United States True
Ivory Coast True
Cape Verde True
